# Task 9: Validation


In [9]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.steady_state import sample_steady_state
from src.ssa_visualization import show_sample_moments

T_END = 1000.0   # far past every relaxation time, so the ODE has settled

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---
## 1. Convergence: SSA → ODE as $N_{\text{rep}} \to \infty$

Draw steady-state samples for growing $N_{\text{rep}}$ and track the absolute
error against the ODE reference. It should fall like $1/\sqrt{N_{\text{rep}}}$.

In [10]:
kin = dict(k_on=0.1, k_off=0.1, k_syn=10.0, k_deg=0.2)


_, y = solve_ode_moments(**kin, t0=0.0, g0=0, r0=0, t_end=T_END)
ode_mu_G, ode_mu_R, _, ode_cov = y[:, -1]

n_rep_values = [100, 1000, 10000, 100000, 1000000]
errors = {"mu_G": [], "mu_R": [], "cov_RG": []}

for n_rep in n_rep_values:
    s = sample_steady_state(**kin, n_rep=n_rep)
    errors["mu_G"].append(abs(s["mu_G"] - ode_mu_G))
    errors["mu_R"].append(abs(s["mu_R"] - ode_mu_R))
    errors["cov_RG"].append(abs(s["cov_RG"] - ode_cov))

pd.DataFrame(errors, index=n_rep_values).rename_axis("n_rep").round(4)

,mu_G,mu_R,cov_RG
n_rep,,,
100,0.0500,1.0300,0.2667
1000,0.0150,0.5890,0.3024
10000,0.0033,0.0117,0.0276
100000,0.0041,0.0324,0.0113
1000000,0.0004,0.0014,0.0050


In [11]:
fig = go.Figure()
for key in ["mu_R", "mu_G", "cov_RG"]:
    fig.add_trace(go.Scatter(x=n_rep_values, y=errors[key],
                             mode="lines+markers", name=f"|Δ {key}|"))

# 1/sqrt(N) guide line
guide = errors["mu_R"][0] * np.sqrt(n_rep_values[0] / np.array(n_rep_values, float))
fig.add_trace(go.Scatter(x=n_rep_values, y=guide, mode="lines",
                         line=dict(dash="dot", color="gray"), name="∝ 1/√N"))

fig.update_layout(template="plotly_white", title="<b>Convergence: |SSA − ODE|</b>",
                  width=700, height=400)
fig.update_xaxes(title_text="n_rep", type="log")
fig.update_yaxes(title_text="absolute error", type="log")
fig.show()

---
## 2. Steady-state moments vs the ODE

For each regime, draw a large SSA sample and compare against the settled ODE
value $(\mu_G, \mu_R, C_{RG})$ from `solve_ode_moments`.

In [12]:
test_cases = {
    "Slow switching":  dict(k_on=0.01, k_off=0.02, k_syn=10.0, k_deg=0.2),
    "Fast switching":  dict(k_on=2.0,  k_off=4.0,  k_syn=10.0, k_deg=0.2),
    "High expression": dict(k_on=0.1,  k_off=0.1,  k_syn=40.0, k_deg=0.5),
    "Low expression":  dict(k_on=0.1,  k_off=0.1,  k_syn=2.0,  k_deg=1.0),
}

rows = []
for name, p in test_cases.items():
    s = sample_steady_state(**p, n_rep=20000)
    _, y = solve_ode_moments(**p, t0=0.0, g0=0, r0=0, t_end=T_END)
    ode_mu_G, ode_mu_R, _, ode_cov = y[:, -1]

    rows.append({"regime": name, "moment": "mu_G",   "SSA": s["mu_G"],   "ODE": ode_mu_G})
    rows.append({"regime": name, "moment": "mu_R",   "SSA": s["mu_R"],   "ODE": ode_mu_R})
    rows.append({"regime": name, "moment": "cov_RG", "SSA": s["cov_RG"], "ODE": ode_cov})

df = pd.DataFrame(rows)
df["rel. error"] = (df["SSA"] - df["ODE"]).abs() / df["ODE"].abs().clip(lower=1e-9)
df.round(3)

,regime,moment,SSA,ODE,rel. error
0,Slow switching,mu_G,0.331,0.333,0.006
1,Slow switching,mu_R,16.595,16.667,0.004
2,Slow switching,cov_RG,9.572,9.662,0.009
3,Fast switching,mu_G,0.330,0.333,0.010
4,Fast switching,mu_R,16.641,16.667,0.002
5,Fast switching,cov_RG,0.334,0.359,0.067
6,High expression,mu_G,0.500,0.500,0.000
7,High expression,mu_R,40.244,40.000,0.006
8,High expression,cov_RG,14.347,14.286,0.004
9,Low expression,mu_G,0.500,0.500,0.000


---
## 3. Edge cases

The ODE reference isn't defined here ($k_{\text{deg}}=0$ has no steady state;
$k_{\text{off}}=0$ breaks `sample_steady_state`'s Beta parameter), so we use the
dynamic SSA `simulate_telegraph` directly.

### 3a. Constitutive expression ($k_{\text{off}}=0$, $G_0=1$)

Gene never switches off → pure birth–death → Poisson statistics
($\langle R\rangle = k_{\text{syn}}/k_{\text{deg}} = 50$, Fano $=1$).

In [13]:
data_const = simulate_telegraph(k_on=0.1, k_off=0.0, k_syn=10.0, k_deg=0.2,
                                t0=0.0, g0=1, r0=0, n_sim=4000, n_rep=500)
m_const = compute_sample_moments(data_const)

tail = slice(-len(m_const["time"]) // 5, None)   # last 20% of the trajectory
mu_R = m_const["mu_R"][tail].mean()
fano = (m_const["sigma_R"][tail] ** 2).mean() / mu_R

print(f"<R>  = {mu_R:.1f}   (expect 50)")
print(f"Fano = {fano:.3f}  (expect 1.0 for Poisson)")
assert abs(fano - 1.0) < 0.25

<R>  = 50.5   (expect 50)
Fano = 1.003  (expect 1.0 for Poisson)


In [14]:
show_sample_moments(m_const, title="Edge case: constitutive (k_off=0)")

### 3b. No degradation ($k_{\text{deg}}=0$)

mRNA is never removed → $\langle R\rangle$ grows linearly with slope
$k_{\text{syn}}\,\mu_G = 5.0 \times 0.5 = 2.5$.

In [15]:
data_nodeg = simulate_telegraph(k_on=0.5, k_off=0.5, k_syn=5.0, k_deg=0.0,
                                t0=0.0, g0=1, r0=0, n_sim=3000, n_rep=500)
m_nodeg = compute_sample_moments(data_nodeg)

tail = slice(-len(m_nodeg["time"]) // 5, None)
slope = np.polyfit(m_nodeg["time"][tail], m_nodeg["mu_R"][tail], 1)[0]

print(f"dR/dt = {slope:.2f}  (expect 2.5)")
assert abs(slope - 2.5) / 2.5 < 0.20

dR/dt = 2.49  (expect 2.5)


In [16]:
show_sample_moments(m_nodeg, title="Edge case: no degradation (k_deg=0)")